# 02 — Modelos básicos no E7: 30 aplicações históricas

**Pergunta:** quando controlamos o dispositivo e usamos uma referência local de PBS, modelos simples conseguem distinguir as preparações de MCF10A e HCC1806?

- **30 aplicações físicas** (15 por linhagem) no E7; cada gota foi medida três vezes e resumida. Dados históricos OLD (6), T01–T08 (8), V2 (12) e V3 (4).
- As campanhas T01–T08 e V3 já foram utilizadas no desenvolvimento histórico. Não são o novo teste final.
- 7 bandas fixas de **diferença de fase entre VE e PBS local**, de 0,1 Hz a 1 MHz.
- **LOBO** deixa um bloco cronológico fora; **LOCO** deixa uma campanha inteira fora; LOO apenas uma gota.
- Comparamos regressão logística, KNN, Random Forest e classificador de um termo inspirado em SISSO. **O SISSO oficial NÃO foi executado.**
- Nenhum resultado será usado para reotimizar o classificador congelado nem para decidir ajustes com base no teste final.
- Só `plt.show()`; figuras não são salvas automaticamente.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import LeaveOneGroupOut, LeaveOneOut
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
from modelos_basicos import basic_models, cv_predictions
plt.rcParams.update({'figure.figsize':(9,5),'font.size':11,'axes.spines.top':False,'axes.spines.right':False})

from sklearn.metrics import classification_report
CSV=Path('dados/E7_30_aplicacoes_historicas_caracteristicas.csv')
df=pd.read_csv(CSV)
features=[f'F_b{i}' for i in range(1,8)]
X=df[features].to_numpy(float)
y=(df.classe=='HCC1806').astype(int).to_numpy()
assert X.shape==(30,7) and sum(y)==15
print(pd.crosstab(df.campanha,df.classe,margins=True).to_string())
print('Blocos:',df.bloco.nunique(),'| Campanhas:',df.campanha.nunique())

## Comparação fixa dos modelos (LOBO, LOCO e LOO)

O escalonamento e a seleção de eventual descritor simbólico ocorrem **somente dentro da partição de treinamento**. Nenhum modelo acessa rótulos do bloco/campanha deixado de fora.

In [ ]:
scenarios=[('LOBO',LeaveOneGroupOut(),df.bloco.to_numpy()),
           ('LOCO',LeaveOneGroupOut(),df.campanha.to_numpy()),
           ('LOO',LeaveOneOut(),None)]
rows=[];predictions={}
for name,model in basic_models().items():
    for valid,split,g in scenarios:
        p=cv_predictions(X,y,split,g,model)
        predictions[(name,valid)]=p
        rows.append({'Modelo':name,'Validação':valid,'Acertos':int((p==y).sum()),'Total':len(y),'Acurácia':(p==y).mean()})
results=pd.DataFrame(rows)
print(results.to_string(index=False,formatters={'Acurácia':'{:.1%}'.format}))
names=list(basic_models())
fig,ax=plt.subplots(figsize=(12,5.1)); x=np.arange(len(names)); w=.24
for offset,valid in zip([-w,0,w],['LOBO','LOCO','LOO']):
    vals=[results.query('Modelo==@name and Validação==@valid')['Acurácia'].iloc[0] for name in names]
    bars=ax.bar(x+offset,100*np.array(vals),w,label=valid)
    for rr,v in zip(bars,vals): ax.text(rr.get_x()+rr.get_width()/2,rr.get_height()+1.4,f'{v:.0%}',ha='center',fontsize=8)
ax.set(xticks=x,xticklabels=names,ylim=(0,115),ylabel='Acurácia (%)',title='E7 — 30 aplicações | modelos simples e validação agrupada')
ax.axhline(50,color='gray',ls='--',lw=1,alpha=.7)
ax.tick_params(axis='x',rotation=10);ax.legend();ax.grid(axis='y',alpha=.18)
plt.tight_layout();plt.show()

## Erros por campanha (LOCO) e por bloco (LOBO)

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12.3,4.7))
for ax,valid,gcol in zip(axes,['LOBO','LOCO'],['bloco','campanha']):
    key='KNN (k=5)'
    pred=predictions[(key,valid)]
    tmp=df[[gcol]].copy();tmp['correct']=(pred==y)
    v=tmp.groupby(gcol).correct.agg(['sum','size','mean'])
    bars=ax.bar(np.arange(len(v)),v['mean']*100)
    ax.set(xticks=np.arange(len(v)),xticklabels=v.index,ylim=(0,115),ylabel='Acerto (%)',title=f'{key}: {valid}')
    ax.tick_params(axis='x',rotation=45)
    for b,(_,r) in zip(bars,v.iterrows()):ax.text(b.get_x()+b.get_width()/2,b.get_height()+2,f'{int(r["sum"])}/{int(r["size"])}',ha='center',fontsize=9)
    ax.grid(axis='y',alpha=.2)
plt.tight_layout();plt.show()

## Matriz de confusão do KNN em LOBO e LOCO

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(9.5,4.2))
for ax,valid in zip(axs,['LOBO','LOCO']):
    cm=confusion_matrix(y,predictions[('KNN (k=5)',valid)],labels=[0,1])
    ax.imshow(cm,cmap='Greys',vmin=0,vmax=15)
    ax.set_xticks([0,1],['MCF10A','HCC1806']);ax.set_yticks([0,1],['MCF10A','HCC1806'])
    for (i,j),v in np.ndenumerate(cm): ax.text(j,i,str(v),ha='center',va='center',fontsize=17,color='white' if v>9 else 'black')
    ax.set(xlabel='Previsto',ylabel='Real',title=f'KNN · {valid}')
plt.tight_layout();plt.show()

## PCA descritiva das mesmas 30 gotas

PCA **não é classificador**. Não utiliza rótulos para ajustar componentes, mas o aspecto visual de separação entre grupos não equivale a acurácia de previsão. Avaliar também a distribuição por campanha para explorar efeitos de lote.

In [ ]:
Xscaled=StandardScaler().fit_transform(X)
pca=PCA(n_components=2)
pcs=pca.fit_transform(Xscaled)
fig,axs=plt.subplots(1,2,figsize=(12,5))
for label in ['MCF10A','HCC1806']:
    idx=(df.classe==label).to_numpy()
    axs[0].scatter(pcs[idx,0],pcs[idx,1],s=75,alpha=.85,label=label)
axs[0].set_title('Cor = linhagem');axs[0].legend()
for camp in df.campanha.unique():
    idx=(df.campanha==camp).to_numpy()
    axs[1].scatter(pcs[idx,0],pcs[idx,1],s=75,alpha=.85,label=camp)
axs[1].set_title('Mesma PCA: cor = campanha');axs[1].legend(title='Campanha')
for ax in axs:
    ax.set(xlabel=f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',ylabel=f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
    ax.axhline(0,color='gray',lw=.7);ax.axvline(0,color='gray',lw=.7);ax.grid(alpha=.16)
fig.suptitle('E7: PCA das sete bandas de fase normalizada pelo PBS')
plt.tight_layout();plt.show()

## Leitura para o TCC

- **PCA** caracteriza a estrutura multivariada; os quatro classificadores estimam o desempenho de discriminação.
- A escolha entre LOBO e LOCO deve corresponder à pergunta científica, e as estimativas sobre as mesmas 30 gotas permanecem exploratórias.
- Modelos não foram otimizados neste notebook. Diferenças entre eles não provam superioridade generalizável com n=30.
- **SISSO-inspirado** só ilustra a ideia de triagem de expressão com um termo. Não escrever “SISSO 3.5 executado” ou compará-lo como se fosse o algoritmo original.
- O novo conjunto de teste prospectivo está completamente fora dos dados, código e resultados.